In [ ]:

from google.colab import drive
drive.mount('/content/drive')


[2]
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve, precision_recall_curve, auc
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import learning_curve
import joblib
[3]
data = pd. read_csv('/content/drive/MyDrive/time_series-hand2.csv')
[4]
data.describe().transpose()

[5]
important_features = ['LV', 'LaA', 'LoA', 'LaD', 'LoD', 'anxiety','TA','LaJ','LoJ','TJ','pid','Trial']
plt.figure(figsize=(20, 20))
data=data.drop(data.index[89125:90392])
data = data[important_features].dropna(subset=['anxiety'])
data=data.reset_index(drop=True)
data['LaD']=data['LaD'].replace(-1,np.nan)
data['LaD']=(data['LaD']-data['LaD'].min())/(+data['LaD'].max()-data['LaD'].min())
data['LaD']=data['LaD'].fillna(0)
data['LoD']=data['LoD'].replace(-1,np.nan)
data['LoD']=(data['LoD']-data['LoD'].min())/(data['LoD'].max()-data['LoD'].min())
data['LoD']=data['LoD'].fillna(1)
sns.heatmap(data.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()
def reduce_classes(x):
    if x in [1]:
        return 1
    elif x in [2]:
        return 2
    elif x in [3,4]:
        return 3
    elif x in [5,6]:
        return 4
    elif x in [7,8]:
        return 5
data['anxiety'] = data['anxiety'].map(reduce_classes)


[7]
class AnxietyPredictor:
    """
    A class for predicting anxiety levels based on various physiological features.
    """
    def __init__(self, data):
        """
        Initialize the AnxietyPredictor with the given data.

        :param data: A pandas DataFrame containing the data.
        """
        self.data = data

    def preprocess_data(self):
        """
        Preprocess the data by separating features and target, encoding the target, and splitting into training and test sets.
        """
        self.features = self.data[['pid','LV', 'LaA','LoA', 'LaD', 'LoD']]
        self.target = self.data['anxiety']

        self.label_encoder = LabelEncoder()
        self.target_encoded = self.label_encoder.fit_transform(self.target)
        self.features['pid'] = self.label_encoder.fit_transform(self.features['pid'])
        print(self.features['pid'])

        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(self.features, self.target_encoded, test_size=0.2)
        self.pid_train = self.X_train['pid']
        self.X_train = self.X_train.drop('pid', axis=1)
        self.pid_test = self.X_test['pid']
        self.X_test = self.X_test.drop('pid', axis=1)
        self.scaler = StandardScaler()
        self.X_train_scaled0 = self.scaler.fit_transform(self.X_train)
        self.X_test_scaled0 = self.scaler.transform(self.X_test)
        self.X_train_scaled =  np.column_stack((self.X_train_scaled0 , self.pid_train))
        self.X_test_scaled =  np.column_stack((self.X_test_scaled0, self.pid_test))





    def train_classifier(self):
        """
        Train a classifier to predict anxiety levels using all features.
        """
        self.classifier = RandomForestClassifier()
        self.classifier.fit(self.X_train_scaled, self.y_train)

    def evaluate_classifier(self):
        """
        Evaluate the performance of the classifier.
        """
        self.y_pred = self.classifier.predict(self.X_test_scaled)
        self.accuracy = accuracy_score(self.y_test, self.y_pred)
        print(f'Accuracy: {self.accuracy:.2f}')

    def save_model(self, model_path, scaler_path):
        joblib.dump(self.classifier, model_path)
        joblib.dump(self.scaler, scaler_path)
        print(f'Model saved to {model_path}')
        print(f'Scaler saved to {scaler_path}')

data = data
predictor = AnxietyPredictor(data)
predictor.preprocess_data()
predictor.train_classifier()
predictor.evaluate_classifier()
model_path = '/content/drive/MyDrive/random_farest_model.joblib'
scaler_path = '/content/drive/MyDrive/random_farest_scaler.joblib'
predictor.save_model(model_path, scaler_path)

[9]
sns.set(style="whitegrid")
plt.rcParams.update({'font.size': 12})

def plot_confusion_matrix(y_test, y_pred):
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=range(cm.shape[1]), yticklabels=range(cm.shape[0]))
    plt.xlabel('Predicted Label', fontsize=14)
    plt.ylabel('True Label', fontsize=14)
    plt.title('Confusion Matrix', fontsize=16)
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.show()

def plot_classification_report(y_test, y_pred):
    cr = classification_report(y_test, y_pred, output_dict=True)
    cr_df = pd.DataFrame(cr).transpose()
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.axis('tight')
    ax.axis('off')
    table = ax.table(cellText=cr_df.values.round(2),
                     colLabels=cr_df.columns,
                     rowLabels=cr_df.index,
                     cellLoc='center',
                     loc='center',
                     colColours=['#f2f2f2']*len(cr_df.columns),
                     rowColours=['#f2f2f2']*len(cr_df))
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1.2, 1.2)
    plt.title('Classification Report', fontsize=16)
    plt.show()

def plot_learning_curve(classifier, X_train_scaled, y_train):
    train_sizes, train_scores, val_scores = learning_curve(classifier, X_train_scaled, y_train, cv=5, scoring='accuracy', n_jobs=-1)
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    val_mean = np.mean(val_scores, axis=1)
    val_std = np.std(val_scores, axis=1)
    plt.figure(figsize=(8, 6))
    plt.plot(train_sizes, train_mean, label='Training Accuracy', color='blue')
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
    plt.plot(train_sizes, val_mean, label='Validation Accuracy', color='red')
    plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='red')
    plt.xlabel('Training Examples')
    plt.ylabel('Accuracy')
    plt.title('Learning Curve')
    plt.legend(loc='best')
    plt.show()


def plot_roc_auc_curve(y_test, y_pred_proba):
    aucs = []
    plt.figure(figsize=(8, 6))
    for i in range(y_pred_proba.shape[1]):
        fpr, tpr, _ = roc_curve(y_test == i, y_pred_proba[:, i])
        auc_score = roc_auc_score(y_test == i, y_pred_proba[:, i])
        aucs.append(auc_score)
        plt.plot(fpr, tpr, label=f"Class {i}, AUC: {auc_score:.3f}")
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel("False Positive Rate", fontsize=14)
    plt.ylabel("True Positive Rate", fontsize=14)
    plt.title("ROC-AUC Curve", fontsize=16)
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_precision_recall_curve(y_test, y_pred_proba):
    le = LabelEncoder()
    y_test_encoded = le.fit_transform(y_test)
    plt.figure(figsize=(8, 6))
    for i in range(y_pred_proba.shape[1]):
        precision, recall, _ = precision_recall_curve(y_test_encoded == i, y_pred_proba[:, i])
        plt.plot(recall, precision, label=f"Class {i}")
    plt.xlabel("Recall", fontsize=14)
    plt.ylabel("Precision", fontsize=14)
    plt.title("Precision-Recall Curve", fontsize=16)
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_class_distribution(y_test):
    plt.figure(figsize=(8, 6))
    sns.countplot(x=y_test, hue=y_test, palette="viridis", dodge=False)
    plt.xlabel("Class", fontsize=14)
    plt.ylabel("Count", fontsize=14)
    plt.title("Class Distribution", fontsize=16)
    plt.grid(True)
    plt.show()


plot_confusion_matrix(predictor.y_test, predictor.y_pred)
print("-" * 80)
plot_classification_report(predictor.y_test, predictor.y_pred)
print("-" * 80)
plot_learning_curve(predictor.classifier, predictor.X_train_scaled, predictor.y_train)
print("-" * 80)
plot_roc_auc_curve(predictor.y_test, predictor.classifier.predict_proba(predictor.X_test_scaled))
print("-" * 80)
plot_precision_recall_curve(predictor.y_test, predictor.classifier.predict_proba(predictor.X_test_scaled))
print("-" * 80)
plot_class_distribution(predictor.y_test)




In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import learning_curve

# Load the data
data = pd.read_csv('/content/drive/MyDrive/time_series-hand2.csv')

# Select important features and visualize correlation matrix
important_features = ['LV', 'LaA', 'LoA', 'LaD', 'LoD', 'anxiety', 'TA', 'LaJ', 'LoJ', 'TJ', 'pid']
data = data[important_features].dropna(subset=['anxiety'])

# Reduce anxiety classes
def reduce_classes(x):
    if x in [1]:
        return 1
    elif x in [2]:
        return 2
    elif x in [3, 4]:
        return 3
    elif x in [5, 6]:
        return 4
    elif x in [7, 8]:
        return 5

data['anxiety'] = data['anxiety'].map(reduce_classes)

# Class for Anxiety Predictor
class AnxietyPredictor:
    """
    A class for predicting anxiety levels based on various physiological features.
    """
    def __init__(self, data):
        """
        Initialize the AnxietyPredictor with the given data.

        :param data: A pandas DataFrame containing the data.
        """
        self.data = data

    def preprocess_data(self):
        """
        Preprocess the data by separating features and target, encoding the target, and splitting into training and test sets.
        """
        # Features and target
        self.features = self.data[['LV', 'LaA', 'LoA', 'LaD', 'LoD', 'pid']]
        self.target = self.data['anxiety']

        # Encode target variable
        self.label_encoder = LabelEncoder()
        self.target_encoded = self.label_encoder.fit_transform(self.target)
        self.features['pid'] = self.label_encoder.fit_transform(self.features['pid'])

        # Split into training and test sets
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(self.features, self.target_encoded, test_size=0.2, random_state=42)
        self.pid_train = self.X_train['pid']
        self.X_train = self.X_train.drop('pid', axis=1)
        self.pid_test = self.X_test['pid']
        self.X_test = self.X_test.drop('pid', axis=1)
        self.scaler = StandardScaler()
        self.X_train_scaled0 = self.scaler.fit_transform(self.X_train)
        self.X_test_scaled0 = self.scaler.transform(self.X_test)
        self.X_train_scaled =  np.column_stack((self.X_train_scaled0 , self.pid_train))
        self.X_test_scaled =  np.column_stack((self.X_test_scaled0, self.pid_test))




        # Standardize features
        self.scaler = StandardScaler()
        self.X_train_scaled = self.scaler.fit_transform(self.X_train)
        self.X_test_scaled = self.scaler.transform(self.X_test)

    def train_random_forest_classifier(self):
        """
        Train a Random Forest classifier to predict anxiety levels.
        """
        self.classifier = RandomForestClassifier()
        self.classifier.fit(self.X_train_scaled, self.y_train)

    def train_svm_classifier(self):
        """
        Train a Support Vector Machine (SVM) classifier to predict anxiety levels.
        """
        self.classifier = SVC(kernel='rbf', random_state=42)
        self.classifier.fit(self.X_train_scaled, self.y_train)

    def train_knn_classifier(self):
        """
        Train a K-Nearest Neighbors (KNN) classifier to predict anxiety levels.
        """
        self.classifier = KNeighborsClassifier(n_neighbors=5)
        self.classifier.fit(self.X_train_scaled, self.y_train)

    def train_decision_tree_classifier(self):
        """
        Train a Decision Tree classifier to predict anxiety levels.
        """
        self.classifier = DecisionTreeClassifier(random_state=42)
        self.classifier.fit(self.X_train_scaled, self.y_train)

    def train_naive_bayes_classifier(self):
        """
        Train a Naive Bayes classifier to predict anxiety levels.
        """
        self.classifier = GaussianNB()
        self.classifier.fit(self.X_train_scaled, self.y_train)

    def evaluate_classifier(self):
        """
        Evaluate the performance of the classifier.
        """
        self.y_pred = self.classifier.predict(self.X_test_scaled)
        self.accuracy = accuracy_score(self.y_test, self.y_pred)
        print(f'Accuracy: {self.accuracy:.2f}')

        # Confusion matrix
        self.conf_matrix = confusion_matrix(self.y_test, self.y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(self.conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=self.classifier.classes_, yticklabels=self.classifier.classes_)
        plt.xlabel('Predicted Label', fontsize=14)
        plt.ylabel('True Label', fontsize=14)
        plt.title('Confusion Matrix', fontsize=16)
        plt.xticks(rotation=45)
        plt.yticks(rotation=0)
        plt.show()

# Instantiate the predictor and perform operations for each classifier
data = data
predictor = AnxietyPredictor(data)

# Random Forest
print("Random Forest Classifier:")
predictor.preprocess_data()
predictor.train_random_forest_classifier()
predictor.evaluate_classifier()

# SVM
print("Support Vector Machine (SVM) Classifier:")
predictor.preprocess_data()
predictor.train_svm_classifier()
predictor.evaluate_classifier()

# KNN
print("K-Nearest Neighbors (KNN) Classifier:")
predictor.preprocess_data()
predictor.train_knn_classifier()
predictor.evaluate_classifier()

# Decision Tree
print("Decision Tree Classifier:")
predictor.preprocess_data()
predictor.train_decision_tree_classifier()
predictor.evaluate_classifier()

# Naive Bayes
print("Naive Bayes Classifier:")
predictor.preprocess_data()
predictor.train_naive_bayes_classifier()
predictor.evaluate_classifier()

In [ ]:
# Importing additional classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier

# Existing imports
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve, precision_recall_curve, auc
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import learning_curve

# Loading the data
data = pd.read_csv('/content/drive/MyDrive/time_series-hand2.csv')

# Data description
data.describe().transpose()

# Selecting important features and creating correlation matrix
important_features = ['LV', 'LaA', 'LoA', 'LaD', 'LoD', 'anxiety','TA','LaJ','LoJ','TJ','pid']
plt.figure(figsize=(20, 20))
data = data[important_features].dropna(subset=['anxiety'])
sns.heatmap(data.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

# Reducing classes
def reduce_classes(x):
    if x in [1]:
        return 1
    elif x in [2]:
        return 2
    elif x in [3,4]:
        return 3
    elif x in [5,6]:
        return 4
    elif x in [7,8]:
        return 5
data['anxiety'] = data['anxiety'].map(reduce_classes)

# AnxietyPredictor class
class AnxietyPredictor:
    """
    A class for predicting anxiety levels based on various physiological features.
    """
    def __init__(self, data):
        """
        Initialize the AnxietyPredictor with the given data.

        :param data: A pandas DataFrame containing the data.
        """
        self.data = data

    def preprocess_data(self):
        """
        Preprocess the data by separating features and target, encoding the target, and splitting into training and test sets.
        """
        self.features = self.data[['LV', 'LaA', 'LoA', 'LaD', 'LoD','pid']]
        self.target = self.data['anxiety']

        self.label_encoder = LabelEncoder()
        self.target_encoded = self.label_encoder.fit_transform(self.target)
        self.features['pid'] = self.label_encoder.fit_transform(self.features['pid'])

        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(self.features, self.target_encoded, test_size=0.2, random_state=42)
        self.pid_train = self.X_train['pid']
        self.X_train = self.X_train.drop('pid', axis=1)
        self.pid_test = self.X_test['pid']
        self.X_test = self.X_test.drop('pid', axis=1)
        self.scaler = StandardScaler()
        self.X_train_scaled0 = self.scaler.fit_transform(self.X_train)
        self.X_test_scaled0 = self.scaler.transform(self.X_test)
        self.X_train_scaled =  np.column_stack((self.X_train_scaled0 , self.pid_train))
        self.X_test_scaled =  np.column_stack((self.X_test_scaled0, self.pid_test))

    def train_classifier(self, classifier):
        """
        Train a classifier to predict anxiety levels using all features.

        :param classifier: The classifier to train.
        """
        self.classifier = classifier
        self.classifier.fit(self.X_train_scaled, self.y_train)

    def evaluate_classifier(self):
        """
        Evaluate the performance of the classifier.
        """
        self.y_pred = self.classifier.predict(self.X_test_scaled)
        self.accuracy = accuracy_score(self.y_test, self.y_pred)
        print(f'Accuracy: {self.accuracy:.2f}')

# Plot functions
def plot_confusion_matrix(y_test, y_pred):
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=range(cm.shape[1]), yticklabels=range(cm.shape[0]))
    plt.xlabel('Predicted Label', fontsize=14)
    plt.ylabel('True Label', fontsize=14)
    plt.title('Confusion Matrix', fontsize=16)
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.show()

def plot_classification_report(y_test, y_pred):
    cr = classification_report(y_test, y_pred, output_dict=True)
    cr_df = pd.DataFrame(cr).transpose()
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.axis('tight')
    ax.axis('off')
    table = ax.table(cellText=cr_df.values.round(2),
                     colLabels=cr_df.columns,
                     rowLabels=cr_df.index,
                     cellLoc='center',
                     loc='center',
                     colColours=['#f2f2f2']*len(cr_df.columns),
                     rowColours=['#f2f2f2']*len(cr_df))
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1.2, 1.2)
    plt.title('Classification Report', fontsize=16)
    plt.show()

def plot_learning_curve(classifier, X_train_scaled, y_train):
    train_sizes, train_scores, val_scores = learning_curve(classifier, X_train_scaled, y_train, cv=5, scoring='accuracy', n_jobs=-1)
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    val_mean = np.mean(val_scores, axis=1)
    val_std = np.std(val_scores, axis=1)
    plt.figure(figsize=(8, 6))
    plt.plot(train_sizes, train_mean, label='Training Accuracy', color='blue')
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
    plt.plot(train_sizes, val_mean, label='Validation Accuracy', color='red')
    plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='red')
    plt.xlabel('Training Examples')
    plt.ylabel('Accuracy')
    plt.title('Learning Curve')
    plt.legend(loc='best')
    plt.show()

def plot_roc_auc_curve(y_test, y_pred_proba):
    aucs = []
    plt.figure(figsize=(8, 6))
    for i in range(y_pred_proba.shape[1]):
        fpr, tpr, _ = roc_curve(y_test == i, y_pred_proba[:, i])
        auc_score = roc_auc_score(y_test == i, y_pred_proba[:, i])
        aucs.append(auc_score)
        plt.plot(fpr, tpr, label=f"Class {i}, AUC: {auc_score:.3f}")
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel("False Positive Rate", fontsize=14)
    plt.ylabel("True Positive Rate", fontsize=14)
    plt.title("ROC-AUC Curve", fontsize=16)
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_precision_recall_curve(y_test, y_pred_proba):
    le = LabelEncoder()
    y_test_encoded = le.fit_transform(y_test)
    plt.figure(figsize=(8, 6))
    for i in range(y_pred_proba.shape[1]):
        precision, recall, _ = precision_recall_curve(y_test_encoded == i, y_pred_proba[:, i])
        plt.plot(recall, precision, label=f"Class {i}")
    plt.xlabel("Recall", fontsize=14)
    plt.ylabel("Precision", fontsize=14)
    plt.title("Precision-Recall Curve", fontsize=16)
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_class_distribution(y_test):
    plt.figure(figsize=(8, 6))
    sns.countplot(x=y_test, hue=y_test, palette="viridis", dodge=False)
    plt.xlabel("Class", fontsize=14)
    plt.ylabel("Count", fontsize=14)
    plt.title("Class Distribution", fontsize=16)
    plt.grid(True)
    plt.show()
# Function for adding Gaussian noise to both train and test kinematic features
def create_noisy_train_test_data(predictor, noise_std, random_state=42):
    """
    Add Gaussian noise to scaled kinematic features of both train and test data.
    pid is not changed.
    """

    np.random.seed(random_state)

    # Noise for training kinematic features
    train_noise = np.random.normal(
        loc=0,
        scale=noise_std,
        size=predictor.X_train_scaled0.shape
    )

    # Noise for test kinematic features
    test_noise = np.random.normal(
        loc=0,
        scale=noise_std,
        size=predictor.X_test_scaled0.shape
    )

    # Add noise to kinematic features
    X_train_noisy0 = predictor.X_train_scaled0 + train_noise
    X_test_noisy0 = predictor.X_test_scaled0 + test_noise

    # Add pid again without noise
    X_train_noisy = np.column_stack(
        (X_train_noisy0, predictor.pid_train)
    )

    X_test_noisy = np.column_stack(
        (X_test_noisy0, predictor.pid_test)
    )

    return X_train_noisy, X_test_noisy


# Training and evaluating multiple classifiers
classifiers = {
    'Random Forest': RandomForestClassifier(),

}

data = data
predictor = AnxietyPredictor(data)
predictor.preprocess_data()

for name, classifier in classifiers.items():
    print(f"Evaluating {name}")
    predictor.train_classifier(classifier)
    predictor.evaluate_classifier()
    print("-" * 80)
    plot_confusion_matrix(predictor.y_test, predictor.y_pred)
    print("-" * 80)
    plot_classification_report(predictor.y_test, predictor.y_pred)
    print("-" * 80)
    plot_learning_curve(predictor.classifier, predictor.X_train_scaled, predictor.y_train)
    print("-" * 80)
    plot_roc_auc_curve(predictor.y_test, predictor.classifier.predict_proba(predictor.X_test_scaled))
    print("-" * 80)
    plot_precision_recall_curve(predictor.y_test, predictor.classifier.predict_proba(predictor.X_test_scaled))
    print("-" * 80)
    plot_class_distribution(predictor.y_test)
        # Training and testing with Gaussian noisy data
    noise_levels = [0.01, 0.05]

    for noise_std in noise_levels:
        print("-" * 80)
        print(f"Training and Testing {name} with Gaussian noise STD = {noise_std}")

        # Create noisy train and noisy test data
        X_train_noisy, X_test_noisy = create_noisy_train_test_data(
            predictor,
            noise_std=noise_std,
            random_state=42
        )

        # Create a new classifier with the same settings
        noisy_classifier = classifier.__class__(**classifier.get_params())

        # Train on noisy training data
        noisy_classifier.fit(X_train_noisy, predictor.y_train)

        # Test on noisy test data
        y_pred_noisy = noisy_classifier.predict(X_test_noisy)

        # Accuracy
        noisy_accuracy = accuracy_score(predictor.y_test, y_pred_noisy)

        print(f"Noisy Train + Noisy Test Accuracy with STD={noise_std}: {noisy_accuracy:.4f}")

        print("-" * 80)
        print(f"Confusion Matrix - Noise STD = {noise_std}")
        plot_confusion_matrix(predictor.y_test, y_pred_noisy)

        print("-" * 80)
        print(f"Classification Report - Noise STD = {noise_std}")
        plot_classification_report(predictor.y_test, y_pred_noisy)

        print("-" * 80)
        y_pred_proba_noisy = noisy_classifier.predict_proba(X_test_noisy)
        plot_roc_auc_curve(predictor.y_test, y_pred_proba_noisy)

        print("-" * 80)
        plot_precision_recall_curve(predictor.y_test, y_pred_proba_noisy)



In [ ]:
# Importing additional classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier

# Existing imports
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve, precision_recall_curve, auc
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import learning_curve

# Loading the data
data = pd.read_csv('/content/drive/MyDrive/time_series-hand2.csv')

# Data description
data.describe().transpose()

# Selecting important features and creating correlation matrix
important_features = ['LV', 'LaA', 'LoA', 'LaD', 'LoD', 'anxiety','TA','LaJ','LoJ','TJ','pid']
plt.figure(figsize=(20, 20))
data = data[important_features].dropna(subset=['anxiety'])
sns.heatmap(data.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

# Reducing classes
def reduce_classes(x):
    if x in [1]:
        return 1
    elif x in [2]:
        return 2
    elif x in [3,4]:
        return 3
    elif x in [5,6]:
        return 4
    elif x in [7,8]:
        return 5
data['anxiety'] = data['anxiety'].map(reduce_classes)

# AnxietyPredictor class
class AnxietyPredictor:
    """
    A class for predicting anxiety levels based on various physiological features.
    """
    def __init__(self, data):
        """
        Initialize the AnxietyPredictor with the given data.

        :param data: A pandas DataFrame containing the data.
        """
        self.data = data

    def preprocess_data(self):
        """
        Preprocess the data by separating features and target, encoding the target, and splitting into training and test sets.
        """
        self.features = self.data[['LV', 'LaA', 'LoA', 'LaD', 'LoD','pid']]
        self.target = self.data['anxiety']

        self.label_encoder = LabelEncoder()
        self.target_encoded = self.label_encoder.fit_transform(self.target)
        self.features['pid'] = self.label_encoder.fit_transform(self.features['pid'])

        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(self.features, self.target_encoded, test_size=0.2, random_state=42)
        self.pid_train = self.X_train['pid']
        self.X_train = self.X_train.drop('pid', axis=1)
        self.pid_test = self.X_test['pid']
        self.X_test = self.X_test.drop('pid', axis=1)
        self.scaler = StandardScaler()
        self.X_train_scaled0 = self.scaler.fit_transform(self.X_train)
        self.X_test_scaled0 = self.scaler.transform(self.X_test)
        self.X_train_scaled =  np.column_stack((self.X_train_scaled0 , self.pid_train))
        self.X_test_scaled =  np.column_stack((self.X_test_scaled0, self.pid_test))

    def train_classifier(self, classifier):
        """
        Train a classifier to predict anxiety levels using all features.

        :param classifier: The classifier to train.
        """
        self.classifier = classifier
        self.classifier.fit(self.X_train_scaled, self.y_train)

    def evaluate_classifier(self):
        """
        Evaluate the performance of the classifier.
        """
        self.y_pred = self.classifier.predict(self.X_test_scaled)
        self.accuracy = accuracy_score(self.y_test, self.y_pred)
        print(f'Accuracy: {self.accuracy:.2f}')

# Plot functions
def plot_confusion_matrix(y_test, y_pred):
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=range(cm.shape[1]), yticklabels=range(cm.shape[0]))
    plt.xlabel('Predicted Label', fontsize=14)
    plt.ylabel('True Label', fontsize=14)
    plt.title('Confusion Matrix', fontsize=16)
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.show()

def plot_classification_report(y_test, y_pred):
    cr = classification_report(y_test, y_pred, output_dict=True)
    cr_df = pd.DataFrame(cr).transpose()
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.axis('tight')
    ax.axis('off')
    table = ax.table(cellText=cr_df.values.round(2),
                     colLabels=cr_df.columns,
                     rowLabels=cr_df.index,
                     cellLoc='center',
                     loc='center',
                     colColours=['#f2f2f2']*len(cr_df.columns),
                     rowColours=['#f2f2f2']*len(cr_df))
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1.2, 1.2)
    plt.title('Classification Report', fontsize=16)
    plt.show()

def plot_learning_curve(classifier, X_train_scaled, y_train):
    train_sizes, train_scores, val_scores = learning_curve(classifier, X_train_scaled, y_train, cv=5, scoring='accuracy', n_jobs=-1)
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    val_mean = np.mean(val_scores, axis=1)
    val_std = np.std(val_scores, axis=1)
    plt.figure(figsize=(8, 6))
    plt.plot(train_sizes, train_mean, label='Training Accuracy', color='blue')
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
    plt.plot(train_sizes, val_mean, label='Validation Accuracy', color='red')
    plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='red')
    plt.xlabel('Training Examples')
    plt.ylabel('Accuracy')
    plt.title('Learning Curve')
    plt.legend(loc='best')
    plt.show()

def plot_roc_auc_curve(y_test, y_pred_proba):
    aucs = []
    plt.figure(figsize=(8, 6))
    for i in range(y_pred_proba.shape[1]):
        fpr, tpr, _ = roc_curve(y_test == i, y_pred_proba[:, i])
        auc_score = roc_auc_score(y_test == i, y_pred_proba[:, i])
        aucs.append(auc_score)
        plt.plot(fpr, tpr, label=f"Class {i}, AUC: {auc_score:.3f}")
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel("False Positive Rate", fontsize=14)
    plt.ylabel("True Positive Rate", fontsize=14)
    plt.title("ROC-AUC Curve", fontsize=16)
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_precision_recall_curve(y_test, y_pred_proba):
    le = LabelEncoder()
    y_test_encoded = le.fit_transform(y_test)
    plt.figure(figsize=(8, 6))
    for i in range(y_pred_proba.shape[1]):
        precision, recall, _ = precision_recall_curve(y_test_encoded == i, y_pred_proba[:, i])
        plt.plot(recall, precision, label=f"Class {i}")
    plt.xlabel("Recall", fontsize=14)
    plt.ylabel("Precision", fontsize=14)
    plt.title("Precision-Recall Curve", fontsize=16)
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_class_distribution(y_test):
    plt.figure(figsize=(8, 6))
    sns.countplot(x=y_test, hue=y_test, palette="viridis", dodge=False)
    plt.xlabel("Class", fontsize=14)
    plt.ylabel("Count", fontsize=14)
    plt.title("Class Distribution", fontsize=16)
    plt.grid(True)
    plt.show()
# Function for adding Gaussian noise to both train and test kinematic features
def create_noisy_train_test_data(predictor, noise_std, random_state=42):
    """
    Add Gaussian noise to scaled kinematic features of both train and test data.
    pid is not changed.
    """

    np.random.seed(random_state)

    # Noise for training kinematic features
    train_noise = np.random.normal(
        loc=0,
        scale=noise_std,
        size=predictor.X_train_scaled0.shape
    )

    # Noise for test kinematic features
    test_noise = np.random.normal(
        loc=0,
        scale=noise_std,
        size=predictor.X_test_scaled0.shape
    )

    # Add noise to kinematic features
    X_train_noisy0 = predictor.X_train_scaled0 + train_noise
    X_test_noisy0 = predictor.X_test_scaled0 + test_noise

    # Add pid again without noise
    X_train_noisy = np.column_stack(
        (X_train_noisy0, predictor.pid_train)
    )

    X_test_noisy = np.column_stack(
        (X_test_noisy0, predictor.pid_test)
    )

    return X_train_noisy, X_test_noisy


# Training and evaluating multiple classifiers
classifiers = {
    'Random Forest': RandomForestClassifier(),

}

data = data
predictor = AnxietyPredictor(data)
predictor.preprocess_data()

for name, classifier in classifiers.items():
    print(f"Evaluating {name}")
    predictor.train_classifier(classifier)
    predictor.evaluate_classifier()
    print("-" * 80)
    plot_confusion_matrix(predictor.y_test, predictor.y_pred)
    print("-" * 80)
    plot_classification_report(predictor.y_test, predictor.y_pred)
    print("-" * 80)
    plot_learning_curve(predictor.classifier, predictor.X_train_scaled, predictor.y_train)
    print("-" * 80)
    plot_roc_auc_curve(predictor.y_test, predictor.classifier.predict_proba(predictor.X_test_scaled))
    print("-" * 80)
    plot_precision_recall_curve(predictor.y_test, predictor.classifier.predict_proba(predictor.X_test_scaled))
    print("-" * 80)
    plot_class_distribution(predictor.y_test)
        # Training and testing with Gaussian noisy data
    noise_levels = [0.01, 0.05]

    for noise_std in noise_levels:
        print("-" * 80)
        print(f"Training and Testing {name} with Gaussian noise STD = {noise_std}")

        # Create noisy train and noisy test data
        X_train_noisy, X_test_noisy = create_noisy_train_test_data(
            predictor,
            noise_std=noise_std,
            random_state=42
        )

        # Create a new classifier with the same settings
        noisy_classifier = classifier.__class__(**classifier.get_params())

        # Train on noisy training data
        noisy_classifier.fit(X_train_noisy, predictor.y_train)

        # Test on noisy test data
        y_pred_noisy = noisy_classifier.predict(X_test_noisy)

        # Accuracy
        noisy_accuracy = accuracy_score(predictor.y_test, y_pred_noisy)

        print(f"Noisy Train + Noisy Test Accuracy with STD={noise_std}: {noisy_accuracy:.4f}")

        print("-" * 80)
        print(f"Confusion Matrix - Noise STD = {noise_std}")
        plot_confusion_matrix(predictor.y_test, y_pred_noisy)

        print("-" * 80)
        print(f"Classification Report - Noise STD = {noise_std}")
        plot_classification_report(predictor.y_test, y_pred_noisy)

        print("-" * 80)
        y_pred_proba_noisy = noisy_classifier.predict_proba(X_test_noisy)
        plot_roc_auc_curve(predictor.y_test, y_pred_proba_noisy)

        print("-" * 80)
        plot_precision_recall_curve(predictor.y_test, y_pred_proba_noisy)



In [ ]:
# Importing additional classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier

# Existing imports
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve, precision_recall_curve, auc
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import learning_curve

# Loading the data
data = pd.read_csv('/content/drive/MyDrive/time_series-hand2.csv')

# Data description
data.describe().transpose()

# Selecting important features and creating correlation matrix
important_features = ['LV', 'LaA', 'LoA', 'LaD', 'LoD', 'anxiety','TA','LaJ','LoJ','TJ','pid']
plt.figure(figsize=(20, 20))
data = data[important_features].dropna(subset=['anxiety'])
sns.heatmap(data.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

# Reducing classes
def reduce_classes(x):
    if x in [1]:
        return 1
    elif x in [2]:
        return 2
    elif x in [3,4]:
        return 3
    elif x in [5,6]:
        return 4
    elif x in [7,8]:
        return 5
data['anxiety'] = data['anxiety'].map(reduce_classes)

# AnxietyPredictor class
class AnxietyPredictor:
    """
    A class for predicting anxiety levels based on various physiological features.
    """
    def __init__(self, data):
        """
        Initialize the AnxietyPredictor with the given data.

        :param data: A pandas DataFrame containing the data.
        """
        self.data = data

    def preprocess_data(self):
        """
        Preprocess the data by separating features and target, encoding the target, and splitting into training and test sets.
        """
        self.features = self.data[['LV', 'LaA', 'LoA', 'LaD', 'LoD','pid']]
        self.target = self.data['anxiety']

        self.label_encoder = LabelEncoder()
        self.target_encoded = self.label_encoder.fit_transform(self.target)
        self.features['pid'] = self.label_encoder.fit_transform(self.features['pid'])

        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(self.features, self.target_encoded, test_size=0.2, random_state=42)
        self.pid_train = self.X_train['pid']
        self.X_train = self.X_train.drop('pid', axis=1)
        self.pid_test = self.X_test['pid']
        self.X_test = self.X_test.drop('pid', axis=1)
        self.scaler = StandardScaler()
        self.X_train_scaled0 = self.scaler.fit_transform(self.X_train)
        self.X_test_scaled0 = self.scaler.transform(self.X_test)
        self.X_train_scaled =  np.column_stack((self.X_train_scaled0 , self.pid_train))
        self.X_test_scaled =  np.column_stack((self.X_test_scaled0, self.pid_test))

    def train_classifier(self, classifier):
        """
        Train a classifier to predict anxiety levels using all features.

        :param classifier: The classifier to train.
        """
        self.classifier = classifier
        self.classifier.fit(self.X_train_scaled, self.y_train)

    def evaluate_classifier(self):
        """
        Evaluate the performance of the classifier.
        """
        self.y_pred = self.classifier.predict(self.X_test_scaled)
        self.accuracy = accuracy_score(self.y_test, self.y_pred)
        print(f'Accuracy: {self.accuracy:.2f}')

# Plot functions
def plot_confusion_matrix(y_test, y_pred):
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=range(cm.shape[1]), yticklabels=range(cm.shape[0]))
    plt.xlabel('Predicted Label', fontsize=14)
    plt.ylabel('True Label', fontsize=14)
    plt.title('Confusion Matrix', fontsize=16)
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.show()

def plot_classification_report(y_test, y_pred):
    cr = classification_report(y_test, y_pred, output_dict=True)
    cr_df = pd.DataFrame(cr).transpose()
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.axis('tight')
    ax.axis('off')
    table = ax.table(cellText=cr_df.values.round(2),
                     colLabels=cr_df.columns,
                     rowLabels=cr_df.index,
                     cellLoc='center',
                     loc='center',
                     colColours=['#f2f2f2']*len(cr_df.columns),
                     rowColours=['#f2f2f2']*len(cr_df))
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1.2, 1.2)
    plt.title('Classification Report', fontsize=16)
    plt.show()

def plot_learning_curve(classifier, X_train_scaled, y_train):
    train_sizes, train_scores, val_scores = learning_curve(classifier, X_train_scaled, y_train, cv=5, scoring='accuracy', n_jobs=-1)
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    val_mean = np.mean(val_scores, axis=1)
    val_std = np.std(val_scores, axis=1)
    plt.figure(figsize=(8, 6))
    plt.plot(train_sizes, train_mean, label='Training Accuracy', color='blue')
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
    plt.plot(train_sizes, val_mean, label='Validation Accuracy', color='red')
    plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='red')
    plt.xlabel('Training Examples')
    plt.ylabel('Accuracy')
    plt.title('Learning Curve')
    plt.legend(loc='best')
    plt.show()

def plot_roc_auc_curve(y_test, y_pred_proba):
    aucs = []
    plt.figure(figsize=(8, 6))
    for i in range(y_pred_proba.shape[1]):
        fpr, tpr, _ = roc_curve(y_test == i, y_pred_proba[:, i])
        auc_score = roc_auc_score(y_test == i, y_pred_proba[:, i])
        aucs.append(auc_score)
        plt.plot(fpr, tpr, label=f"Class {i}, AUC: {auc_score:.3f}")
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel("False Positive Rate", fontsize=14)
    plt.ylabel("True Positive Rate", fontsize=14)
    plt.title("ROC-AUC Curve", fontsize=16)
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_precision_recall_curve(y_test, y_pred_proba):
    le = LabelEncoder()
    y_test_encoded = le.fit_transform(y_test)
    plt.figure(figsize=(8, 6))
    for i in range(y_pred_proba.shape[1]):
        precision, recall, _ = precision_recall_curve(y_test_encoded == i, y_pred_proba[:, i])
        plt.plot(recall, precision, label=f"Class {i}")
    plt.xlabel("Recall", fontsize=14)
    plt.ylabel("Precision", fontsize=14)
    plt.title("Precision-Recall Curve", fontsize=16)
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_class_distribution(y_test):
    plt.figure(figsize=(8, 6))
    sns.countplot(x=y_test, hue=y_test, palette="viridis", dodge=False)
    plt.xlabel("Class", fontsize=14)
    plt.ylabel("Count", fontsize=14)
    plt.title("Class Distribution", fontsize=16)
    plt.grid(True)
    plt.show()
# Function for adding Gaussian noise to test kinematic features
def create_noisy_test_data(predictor, noise_std, random_state=42):
    """
    Add Gaussian noise only to the scaled kinematic test features.
    pid is not changed.
    """

    np.random.seed(random_state)

    noise = np.random.normal(
        loc=0,
        scale=noise_std,
        size=predictor.X_test_scaled0.shape
    )

    X_test_noisy0 = predictor.X_test_scaled0 + noise

    X_test_noisy = np.column_stack(
        (X_test_noisy0, predictor.pid_test)
    )

    return X_test_noisy




# Training and evaluating multiple classifiers
classifiers = {
    'Random Forest': RandomForestClassifier(),

}

data = data
predictor = AnxietyPredictor(data)
predictor.preprocess_data()

for name, classifier in classifiers.items():
    print(f"Evaluating {name}")
    predictor.train_classifier(classifier)
    predictor.evaluate_classifier()
    print("-" * 80)
    plot_confusion_matrix(predictor.y_test, predictor.y_pred)
    print("-" * 80)
    plot_classification_report(predictor.y_test, predictor.y_pred)
    print("-" * 80)
    plot_learning_curve(predictor.classifier, predictor.X_train_scaled, predictor.y_train)
    print("-" * 80)
    plot_roc_auc_curve(predictor.y_test, predictor.classifier.predict_proba(predictor.X_test_scaled))
    print("-" * 80)
    plot_precision_recall_curve(predictor.y_test, predictor.classifier.predict_proba(predictor.X_test_scaled))
    print("-" * 80)
    plot_class_distribution(predictor.y_test)
        # Testing model robustness with Gaussian noise
    noise_levels = [0.01, 0.05]

    for noise_std in noise_levels:
        print("-" * 80)
        print(f"Evaluating {name} with Gaussian noise STD = {noise_std}")

        X_test_noisy = create_noisy_test_data(
            predictor,
            noise_std=noise_std,
            random_state=42
        )

        y_pred_noisy = predictor.classifier.predict(X_test_noisy)

        noisy_accuracy = accuracy_score(predictor.y_test, y_pred_noisy)

        print(f"Noisy Test Accuracy with STD={noise_std}: {noisy_accuracy:.4f}")

        print("-" * 80)
        plot_confusion_matrix(predictor.y_test, y_pred_noisy)

        print("-" * 80)
        plot_classification_report(predictor.y_test, y_pred_noisy)

        print("-" * 80)
        y_pred_proba_noisy = predictor.classifier.predict_proba(X_test_noisy)
        plot_roc_auc_curve(predictor.y_test, y_pred_proba_noisy)

        print("-" * 80)
        plot_precision_recall_curve(predictor.y_test, y_pred_proba_noisy)



